# 01 — Build `tomato_leaf_disease_v1` (Kaggle)

**Mục tiêu cuối cùng của notebook này:** tạo ra **một bộ dataset hoàn chỉnh** `tomato_leaf_disease_v1` (ảnh + label YOLO 4 class + negative sample cho lá khỏe + toàn bộ manifest) sẵn sàng **Save Version thành Kaggle Dataset**.

## Trước khi chạy
1. **Add Input** → attach output đã Save Version của `00_data_acquisition_leaf.ipynb` (chứa `ai/datasets/raw/{leaf_roboflow,leaf_zenodo}`).
2. Panel phải → **Internet: ON** (cần để `pip install imagehash`).
3. Accelerator: **None** — notebook này không train.

## Notebook này làm gì
1. Tự nhận diện thư mục input + khám phá cấu trúc thật của `leaf_roboflow` và `leaf_zenodo` (không đoán mò đường dẫn).
2. Liệt kê **tên class GỐC** thật sự có trong `leaf_roboflow` — đối chiếu với 11 class công khai đã biết trước khi tin tưởng class mapping.
3. Remap 4 bệnh mục tiêu (`leaf_early_blight`, `leaf_late_blight`, `leaf_mold`, `leaf_septoria_spot`); class `Healthy` → **negative sample** (ảnh giữ, label rỗng); các class ngoài phạm vi MVP (Bacterial Spot, Iron Deficiency, Leaf Miner, Mosaic Virus, Spider Mites, Yellow Leaf Curl Virus) → loại box, và loại luôn ảnh nếu ảnh đó không còn box mục tiêu nào và cũng không phải ảnh khỏe thật sự (tránh biến ảnh có bệnh-ngoài-phạm-vi thành negative sai).
4. Phát hiện ảnh trùng/near-duplicate (SHA-256 + pHash) trong `leaf_roboflow` để nhóm ảnh trùng vào cùng 1 split.
5. Chia `train/val/test` (75/15/10) theo nhóm trùng lặp — chống data leakage.
6. Audit chéo với `leaf_zenodo` (số ảnh, class gốc, tỷ lệ trùng SHA-256 với `leaf_roboflow`) để có dữ liệu ra quyết định gộp ở phiên bản sau — **không tự động gộp `leaf_zenodo` vào dataset chính**, đúng theo `HUONG_DAN_XAY_DUNG_DATASET_SMART_GREENHOUSE_AI.md` mục 2.2 (\"Chưa gộp ngay\").
7. Sinh đầy đủ manifest: `data.yaml`, `class_mapping.yaml`, `class_distribution.csv`, `split_manifest.csv`, `duplicate_report.csv`, `rejected_images.csv`, `zenodo_audit.md`, `licenses.md`, `dataset_report.md`.
8. Vẽ mẫu ảnh + bounding box để xác nhận trực quan, gồm cả ví dụ ảnh negative (lá khỏe, không box).

In [ ]:
import subprocess
subprocess.run(["pip", "install", "-q", "imagehash"], check=False)
subprocess.run(["pip", "install", "-q", "pyyaml"], check=False)

In [ ]:
import hashlib
import random
import re
import shutil
import time
from collections import defaultdict
from pathlib import Path

import imagehash
import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from PIL import Image

random.seed(42)
np.random.seed(42)

WORKING = Path("/kaggle/working/ai/datasets")
PROCESSED = WORKING / "processed" / "tomato_leaf_disease_v1"
INTERIM = WORKING / "interim"
MANIFESTS = WORKING / "manifests"

for d in [PROCESSED, INTERIM, MANIFESTS]:
    d.mkdir(parents=True, exist_ok=True)

TARGET_CLASSES = ["leaf_early_blight", "leaf_late_blight", "leaf_mold", "leaf_septoria_spot"]
CLASS_NAME_TO_ID = {name: i for i, name in enumerate(TARGET_CLASSES)}
NEGATIVE = "NEGATIVE_HEALTHY"
OUT_OF_SCOPE = "OUT_OF_SCOPE"

print("PROCESSED:", PROCESSED)

In [ ]:
print("Các thư mục cấp 1 trong /kaggle/input:")
for p in sorted(Path("/kaggle/input").glob("*")):
    print(" -", p)

EXPECTED_SOURCES = ["leaf_roboflow", "leaf_zenodo"]


def find_raw_root():
    hits = sorted(Path("/kaggle/input").glob("*/ai/datasets/raw"))
    if hits:
        return hits[0]
    hits = sorted(Path("/kaggle/input").glob("**/ai/datasets/raw"))
    if hits:
        return hits[0]
    for src_name in EXPECTED_SOURCES:
        hits = [p for p in Path("/kaggle/input").rglob(src_name) if p.is_dir()]
        if hits:
            return hits[0].parent
    return None


# Đặt thủ công nếu auto-detect vẫn chọn sai, ví dụ:
# INPUT_OVERRIDE = Path("/kaggle/input/notebook-xxxx/ai/datasets/raw")
INPUT_OVERRIDE = None

RAW_INPUT = INPUT_OVERRIDE or find_raw_root()

if RAW_INPUT is None:
    raise FileNotFoundError(
        "Không tìm thấy raw data trong /kaggle/input (xem danh sách in phía trên).\n"
        "Kiểm tra: (1) đã Add Input output của 00_data_acquisition_leaf.ipynb chưa, "
        "(2) đã chạy lại notebook (Run All) SAU KHI attach chưa.\n"
        "Nếu cấu trúc thực tế khác thường, đặt INPUT_OVERRIDE ở trên trỏ thẳng tới thư mục "
        "chứa leaf_roboflow/, leaf_zenodo/."
    )

print("\nDùng RAW_INPUT =", RAW_INPUT)

SOURCE_DIRS = {
    "leaf_roboflow": RAW_INPUT / "leaf_roboflow",
    "leaf_zenodo": RAW_INPUT / "leaf_zenodo",
}
for name, p in SOURCE_DIRS.items():
    print(f"  [{'OK' if p.exists() else 'THIẾU'}] {name}: {p}")

### Bước 1 — Khám phá cấu trúc thư mục thật (không đoán mò)
Roboflow export dạng YOLOv8 thường có `train/valid/test` sẵn + `data.yaml` — nhưng ta sẽ **gộp lại và tự chia split** để kiểm soát chống leakage giống nhánh độ chín, không dùng nguyên split có sẵn của Roboflow.

In [ ]:
def ext_counts(root):
    counts = defaultdict(int)
    for p in Path(root).rglob("*"):
        if p.is_file():
            counts[p.suffix.lower()] += 1
    return dict(sorted(counts.items(), key=lambda kv: -kv[1]))


def describe_structure(root, max_depth=3, samples=3):
    root = Path(root)
    if not root.exists():
        print(f"  (không tồn tại: {root})")
        return
    dirs = sorted({p for p in root.rglob("*") if p.is_dir() and len(p.relative_to(root).parts) <= max_depth})
    for d in [root] + dirs:
        depth = len(d.relative_to(root).parts)
        if depth > max_depth:
            continue
        files = sorted(f.name for f in d.iterdir() if f.is_file())
        label = d.relative_to(root) if d != root else "."
        indent = "  " * depth
        print(f"{indent}{label}/  ({len(files)} file)")
        for f in files[:samples]:
            print(f"{indent}  - {f}")
        if len(files) > samples:
            print(f"{indent}  ... và {len(files) - samples} file khác")


for name, path in SOURCE_DIRS.items():
    print(f"\n===== {name} =====")
    print("Đếm file theo đuôi:", ext_counts(path))
    describe_structure(path)

print("\n===== File định nghĩa class (data.yaml/classes.txt) =====")
for name in SOURCE_DIRS:
    root = SOURCE_DIRS[name]
    found = list(root.rglob("data.yaml")) + list(root.rglob("*.yaml")) + list(root.rglob("*.yml"))
    print(f"\n-- {name} --")
    if not found:
        print("  Không tìm thấy data.yaml.")
    for f in found:
        print(" ", f)
        try:
            print("   nội dung:", f.read_text(encoding="utf-8", errors="ignore")[:500])
        except Exception as e:
            print("   (không đọc được:", e, ")")

### Bước 2 — Liệt kê tên class GỐC thật sự có trong dữ liệu
So sánh với `name_map` ở Bước 3. `leaf_roboflow` được công bố có 11 class; nếu tên in ra khác chính tả/hoa-thường so với bảng đã biết, phải sửa `name_map` trước khi tin tưởng — **không remap chỉ dựa vào phỏng đoán**.

In [ ]:
def load_id_to_name(name):
    """Đọc id->name thật từ data.yaml của nguồn YOLO. Không có sẵn -> None (không đoán)."""
    root = SOURCE_DIRS[name]
    for yf in list(root.rglob("data.yaml")) + list(root.rglob("*.yaml")) + list(root.rglob("*.yml")):
        try:
            y = yaml.safe_load(yf.read_text(encoding="utf-8"))
        except Exception:
            continue
        names = y.get("names") if isinstance(y, dict) else None
        if isinstance(names, list):
            return {i: nm for i, nm in enumerate(names)}, yf
        if isinstance(names, dict):
            return {int(k): v for k, v in names.items()}, yf
    return None, None


ID_TO_NAME = {}
for src in SOURCE_DIRS:
    found, src_file = load_id_to_name(src)
    if found:
        print(f"[OK] {src}: id_to_name từ {src_file}:")
        for i, nm in sorted(found.items()):
            print(f"    {i}: {nm}")
        ID_TO_NAME[src] = found
    else:
        raise FileNotFoundError(
            f"Không tìm thấy data.yaml cho {src} -> không thể remap an toàn, dừng lại thay vì đoán."
        )

KNOWN_ROBOFLOW_CLASSES = {
    "Healthy", "Bacterial Spot", "Early Blight", "Iron Deficiency", "Late Blight",
    "Leaf Mold", "Leaf Miner", "Mosaic Virus", "Septoria", "Spider Mites",
    "Yellow Leaf Curl Virus",
}
roboflow_names = set(ID_TO_NAME["leaf_roboflow"].values())
print("\nSo với 11 class công khai đã biết trước:")
print("  Có trong data.yaml nhưng KHÔNG có trong danh sách đã biết:", roboflow_names - KNOWN_ROBOFLOW_CLASSES)
print("  Có trong danh sách đã biết nhưng KHÔNG có trong data.yaml:", KNOWN_ROBOFLOW_CLASSES - roboflow_names)

### Bước 3 — Class mapping chuẩn
Theo `HUONG_DAN_XAY_DUNG_DATASET_SMART_GREENHOUSE_AI.md` mục 2.1. `name_map` dùng đúng tên in ra ở Bước 2 (không phải tên tài liệu tóm tắt) — nếu Bước 2 báo lệch tên, sửa `name_map` này trước khi chạy tiếp.

In [ ]:
def normalize_name(s):
    s = s.strip().lower()
    return re.sub(r"[\s_-]+", "_", s)


# name_map dùng key đã normalize_name() để không phụ thuộc hoa/thường hay khoảng trắng/gạch dưới.
ROBOFLOW_NAME_MAP = {
    normalize_name("Early Blight"): "leaf_early_blight",
    normalize_name("Late Blight"): "leaf_late_blight",
    normalize_name("Leaf Mold"): "leaf_mold",
    normalize_name("Septoria"): "leaf_septoria_spot",
    normalize_name("Healthy"): NEGATIVE,
    normalize_name("Bacterial Spot"): OUT_OF_SCOPE,
    normalize_name("Iron Deficiency"): OUT_OF_SCOPE,
    normalize_name("Leaf Miner"): OUT_OF_SCOPE,
    normalize_name("Mosaic Virus"): OUT_OF_SCOPE,
    normalize_name("Spider Mites"): OUT_OF_SCOPE,
    normalize_name("Yellow Leaf Curl Virus"): OUT_OF_SCOPE,
}

# Cảnh báo nếu có tên thật trong data.yaml không khớp bất kỳ key nào ở trên (tránh âm thầm rơi vào "unmapped").
unmapped_names = [nm for nm in ID_TO_NAME["leaf_roboflow"].values() if normalize_name(nm) not in ROBOFLOW_NAME_MAP]
if unmapped_names:
    print(f"[CẢNH BÁO] Các tên class sau trong data.yaml chưa có trong ROBOFLOW_NAME_MAP -> box sẽ bị coi OUT_OF_SCOPE: {unmapped_names}")
    for nm in unmapped_names:
        ROBOFLOW_NAME_MAP[normalize_name(nm)] = OUT_OF_SCOPE
else:
    print("[OK] Mọi tên class trong data.yaml đều đã có trong ROBOFLOW_NAME_MAP.")

CLASS_MAPPING = {
    "leaf_roboflow": {
        "id_to_name": ID_TO_NAME["leaf_roboflow"],
        "name_map": ROBOFLOW_NAME_MAP,
    },
}

mapping_out = {
    "target_classes": TARGET_CLASSES,
    "negative_marker": NEGATIVE,
    "out_of_scope_marker": OUT_OF_SCOPE,
    **CLASS_MAPPING,
}
with open(MANIFESTS / "class_mapping.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(mapping_out, f, allow_unicode=True, sort_keys=False)

print("\n" + (MANIFESTS / "class_mapping.yaml").read_text(encoding="utf-8"))

### Bước 4 — Parser YOLO + quy tắc quyết định ảnh dương/âm/loại
- Còn box mục tiêu sau remap → ảnh **dương** (positive), giữ các box đó.
- Không còn box mục tiêu, và (không có annotation nào hoặc annotation chỉ toàn `Healthy`) → ảnh **âm** (negative), label rỗng.
- Không còn box mục tiêu nhưng có ít nhất 1 box ngoài phạm vi MVP (bệnh khác, không phải Healthy) → **loại** ảnh (không dùng làm negative sai, vì ảnh thực sự có bệnh, chỉ là bệnh chưa nằm trong MVP).

In [ ]:
def find_images(root):
    exts = {".jpg", ".jpeg", ".png", ".bmp"}
    return {p.stem: p for p in Path(root).rglob("*") if p.suffix.lower() in exts}


def parse_yolo(txt_path, img_w, img_h, id_to_name):
    boxes = []
    for line in Path(txt_path).read_text().splitlines():
        parts = line.split()
        if len(parts) < 5:
            continue
        cid = int(parts[0])
        xc, yc, bw, bh = map(float, parts[1:5])
        cname = id_to_name.get(cid, "UNKNOWN")
        xmin, ymin = (xc - bw / 2) * img_w, (yc - bh / 2) * img_h
        xmax, ymax = (xc + bw / 2) * img_w, (yc + bh / 2) * img_h
        boxes.append((cname, xmin, ymin, xmax, ymax))
    return boxes


def load_source_annotations_yolo(name, id_to_name):
    """stem -> (img_path, img_w, img_h, boxes[(class_name, xmin, ymin, xmax, ymax)])"""
    root = SOURCE_DIRS[name]
    images = find_images(root)
    txt_by_stem = {p.stem: p for p in root.rglob("*.txt")}
    result = {}
    for stem, img_path in images.items():
        with Image.open(img_path) as im:
            w, h = im.size
        txt_path = txt_by_stem.get(stem)
        boxes = parse_yolo(txt_path, w, h, id_to_name) if txt_path else []
        result[stem] = (img_path, w, h, boxes)
    return result


def remap_boxes_leaf(boxes, img_w, img_h, name_map):
    lines, n_healthy, n_outofscope = [], 0, 0
    for cname, xmin, ymin, xmax, ymax in boxes:
        target = name_map.get(normalize_name(cname), OUT_OF_SCOPE)
        if target == OUT_OF_SCOPE:
            n_outofscope += 1
            continue
        if target == NEGATIVE:
            n_healthy += 1
            continue
        xmin, xmax = max(0, xmin), min(img_w, xmax)
        ymin, ymax = max(0, ymin), min(img_h, ymax)
        if xmax <= xmin or ymax <= ymin:
            continue
        xc, yc = (xmin + xmax) / 2 / img_w, (ymin + ymax) / 2 / img_h
        bw, bh = (xmax - xmin) / img_w, (ymax - ymin) / img_h
        lines.append(f"{CLASS_NAME_TO_ID[target]} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}")
    return lines, n_healthy, n_outofscope


def convert_source_leaf(annotations, name_map, out_dir, prefix):
    (out_dir / "images").mkdir(parents=True, exist_ok=True)
    (out_dir / "labels").mkdir(parents=True, exist_ok=True)

    stats = {
        "images_total": len(annotations), "images_kept_positive": 0, "images_kept_negative": 0,
        "images_rejected": 0, "boxes_kept": 0, "boxes_dropped_healthy": 0, "boxes_dropped_outofscope": 0,
    }
    rejected_rows = []

    for stem, (img_path, w, h, boxes) in annotations.items():
        lines, n_healthy, n_outofscope = remap_boxes_leaf(boxes, w, h, name_map)
        stats["boxes_dropped_healthy"] += n_healthy
        stats["boxes_dropped_outofscope"] += n_outofscope

        if lines:
            fate = "positive"
        elif len(boxes) == 0 or (n_healthy > 0 and n_outofscope == 0):
            fate = "negative"
        else:
            fate = "rejected"

        if fate == "rejected":
            rejected_rows.append({"image": img_path.name, "reason": "chi_co_class_ngoai_mvp_khong_phai_healthy"})
            stats["images_rejected"] += 1
            continue

        dest_img = out_dir / "images" / f"{prefix}__{img_path.name}"
        shutil.copy2(img_path, dest_img)
        (out_dir / "labels" / f"{prefix}__{img_path.stem}.txt").write_text("\n".join(lines), encoding="utf-8")
        if fate == "positive":
            stats["images_kept_positive"] += 1
            stats["boxes_kept"] += len(lines)
        else:
            stats["images_kept_negative"] += 1

    return stats, rejected_rows

In [ ]:
annotations = load_source_annotations_yolo("leaf_roboflow", ID_TO_NAME["leaf_roboflow"])
out_dir = INTERIM / "leaf_roboflow_normalized"

print(f"== Convert leaf_roboflow ({len(annotations)} ảnh) ==")
stats, rejected_rows = convert_source_leaf(annotations, ROBOFLOW_NAME_MAP, out_dir, prefix="leaf_roboflow")
print(stats)

pd.DataFrame(rejected_rows).to_csv(MANIFESTS / "rejected_images.csv", index=False)
print(f"\nrejected_images.csv: {len(rejected_rows)} dòng (ảnh chỉ có bệnh ngoài phạm vi MVP, không phải Healthy)")

### Bước 5 — Phát hiện ảnh trùng / near-duplicate trong `leaf_roboflow`
SHA-256 cho trùng chính xác + pHash (Hamming distance ≤ 6) cho near-duplicate, dùng union-find để gom nhóm — giống hệt cách làm ở `01_build_tomato_ripeness_v1.ipynb`.

In [ ]:
def sha256_file(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for c in iter(lambda: f.read(chunk), b""):
            h.update(c)
    return h.hexdigest()


images_dir = out_dir / "images"
all_images = sorted(images_dir.glob("*"))
print(f"Tổng số ảnh sau convert (positive + negative): {len(all_images)}")

records = []
for path in all_images:
    sha = sha256_file(path)
    try:
        phash = str(imagehash.phash(Image.open(path)))
    except Exception:
        phash = None
    records.append({"path": str(path), "filename": path.name, "sha256": sha, "phash": phash})

img_df = pd.DataFrame(records)
print("Đã hash xong", len(img_df), "ảnh.")

In [ ]:
n = len(img_df)
parent = list(range(n))


def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]
        i = parent[i]
    return i


def union(i, j):
    ri, rj = find(i), find(j)
    if ri != rj:
        parent[ri] = rj


sha_groups = defaultdict(list)
for i, sha in enumerate(img_df["sha256"]):
    sha_groups[sha].append(i)
for idxs in sha_groups.values():
    for j in idxs[1:]:
        union(idxs[0], j)

phash_ints = [int(p, 16) if p else None for p in img_df["phash"]]
THRESH = 6
t0 = time.time()
for i in range(n):
    if phash_ints[i] is None:
        continue
    for j in range(i + 1, n):
        if phash_ints[j] is None:
            continue
        if bin(phash_ints[i] ^ phash_ints[j]).count("1") <= THRESH:
            union(i, j)
print(f"So khớp pHash {n}x{n}: {time.time() - t0:.1f}s")

img_df["duplicate_group_id"] = [find(i) for i in range(n)]

dup_report = img_df.groupby("duplicate_group_id").filter(lambda g: len(g) > 1).sort_values("duplicate_group_id")
dup_report.to_csv(MANIFESTS / "duplicate_report.csv", index=False)
print(f"Ảnh nằm trong nhóm trùng/near-duplicate: {len(dup_report)} / {n}")
print(f"Số nhóm trùng: {dup_report['duplicate_group_id'].nunique() if len(dup_report) else 0}")

### Bước 6 — Chia train/val/test chống leakage
75/15/10 theo `duplicate_group_id` (ảnh cùng nhóm trùng luôn nằm cùng 1 split). Áp dụng cho cả ảnh dương lẫn ảnh âm (negative) — không tách riêng, để mỗi split đều có tỷ lệ negative tự nhiên như dữ liệu gốc.

In [ ]:
groups = img_df["duplicate_group_id"].unique().tolist()
random.Random(42).shuffle(groups)
n_g = len(groups)
n_train = int(n_g * 0.75)
n_val = int(n_g * 0.15)

split_assignment = {}
for g in groups[:n_train]:
    split_assignment[g] = "train"
for g in groups[n_train:n_train + n_val]:
    split_assignment[g] = "val"
for g in groups[n_train + n_val:]:
    split_assignment[g] = "test"

img_df["split"] = img_df["duplicate_group_id"].map(split_assignment)


def is_negative(filename):
    label_path = out_dir / "labels" / (Path(filename).stem + ".txt")
    return label_path.exists() and label_path.stat().st_size == 0


img_df["is_negative"] = img_df["filename"].apply(is_negative)
print(img_df.groupby(["split", "is_negative"]).size().unstack(fill_value=0))

for _, row in img_df.iterrows():
    src_img = Path(row["path"])
    src_lbl = out_dir / "labels" / (src_img.stem + ".txt")
    dst_dir = PROCESSED / row["split"]
    (dst_dir / "images").mkdir(parents=True, exist_ok=True)
    (dst_dir / "labels").mkdir(parents=True, exist_ok=True)
    shutil.copy2(src_img, dst_dir / "images" / src_img.name)
    if src_lbl.exists():
        shutil.copy2(src_lbl, dst_dir / "labels" / src_lbl.name)

img_df[["filename", "sha256", "phash", "duplicate_group_id", "split", "is_negative"]].to_csv(
    MANIFESTS / "split_manifest.csv", index=False
)
print("\nĐã copy xong vào", PROCESSED)

### Bước 7 — Audit chéo với `leaf_zenodo` (KHÔNG gộp vào dataset chính)
Theo mục 2.2 của tài liệu, `leaf_zenodo` chỉ được gộp sau khi xác nhận: class mapping rõ ràng, chất lượng label tốt, không trùng lặp nghiêm trọng với `leaf_roboflow`, không gây leakage. Cell này chỉ tính toán các con số đó để quyết định sau, **không ghi ảnh Zenodo vào `PROCESSED`**.

In [ ]:
zenodo_images = find_images(SOURCE_DIRS["leaf_zenodo"])
print(f"leaf_zenodo: {len(zenodo_images)} ảnh, class trong data.yaml: {list(ID_TO_NAME['leaf_zenodo'].values())}")

roboflow_sha_set = set(img_df["sha256"])

t0 = time.time()
zenodo_sha = {stem: sha256_file(p) for stem, p in zenodo_images.items()}
exact_overlap = sum(1 for sha in zenodo_sha.values() if sha in roboflow_sha_set)
print(f"Đã hash {len(zenodo_sha)} ảnh leaf_zenodo trong {time.time() - t0:.1f}s")
print(f"Trùng SHA-256 chính xác với leaf_roboflow (đã giữ trong dataset): {exact_overlap} / {len(zenodo_sha)} "
      f"({100 * exact_overlap / max(len(zenodo_sha), 1):.1f}%)")
print("\nLưu ý: chưa kiểm tra near-duplicate (pHash) chéo giữa 2 nguồn vì chi phí O(n*m) khá lớn "
      "với quy mô 2 bộ này — để lại cho lần audit thủ công nếu quyết định gộp leaf_zenodo sau này.")

zenodo_audit_lines = [
    "# Audit `leaf_zenodo` (chưa gộp vào tomato_leaf_disease_v1)",
    f"- Số ảnh: {len(zenodo_images)}",
    f"- Class trong data.yaml: {list(ID_TO_NAME['leaf_zenodo'].values())}",
    f"- Trùng SHA-256 chính xác với leaf_roboflow: {exact_overlap} ảnh "
    f"({100 * exact_overlap / max(len(zenodo_sha), 1):.1f}%)",
    "- File nguồn đã được Zenodo export sẵn dạng YOLOv8 kèm resize 640x640 + augmentation "
    "(không phải ảnh gốc thuần) — cần cân nhắc khi so chất lượng label.",
    "",
    "## Điều kiện gộp vào phiên bản sau (theo HUONG_DAN_XAY_DUNG_DATASET_SMART_GREENHOUSE_AI.md muc 2.2)",
    "- [ ] Class ánh xạ rõ ràng sang 4 class mục tiêu.",
    "- [ ] Label đạt chất lượng (audit thủ công mẫu ngẫu nhiên).",
    "- [ ] Không trùng lặp nghiêm trọng với leaf_roboflow (SHA-256 ở trên là mức sàn, nên audit thêm pHash).",
    "- [ ] Không gây leakage giữa các split nếu gộp.",
]
(MANIFESTS / "zenodo_audit.md").write_text("\n".join(zenodo_audit_lines), encoding="utf-8")
print("\n" + "\n".join(zenodo_audit_lines))

### Bước 8 — Manifest cuối: `data.yaml`, thống kê class/negative, license, báo cáo

In [ ]:
def count_classes(split_dir):
    counts = defaultdict(int)
    n_negative = 0
    labels_dir = split_dir / "labels"
    if not labels_dir.exists():
        return counts, n_negative
    for txt in labels_dir.glob("*.txt"):
        lines = [l for l in txt.read_text().splitlines() if l.strip()]
        if not lines:
            n_negative += 1
            continue
        for line in lines:
            counts[TARGET_CLASSES[int(line.split()[0])]] += 1
    return counts, n_negative


rows = []
for split_name in ["train", "val", "test"]:
    split_dir = PROCESSED / split_name
    n_images = len(list((split_dir / "images").glob("*"))) if (split_dir / "images").exists() else 0
    class_counts, n_negative = count_classes(split_dir)
    rows.append({"split": split_name, "n_images": n_images, "n_negative": n_negative, **class_counts})

class_dist_df = pd.DataFrame(rows).fillna(0)
class_dist_df.to_csv(MANIFESTS / "class_distribution.csv", index=False)
print(class_dist_df)

data_yaml = {
    "path": str(PROCESSED),
    "train": "train/images",
    "val": "val/images",
    "test": "test/images",
    "names": {i: n for i, n in enumerate(TARGET_CLASSES)},
}
with open(PROCESSED / "data.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(data_yaml, f, allow_unicode=True, sort_keys=False)
print("\n" + (PROCESSED / "data.yaml").read_text())

In [ ]:
licenses_md = "\n".join([
    "# Licenses — tomato_leaf_disease_v1",
    "",
    f"- **Tomato Leaf Disease (Roboflow)** (`leaf_roboflow`): CC BY 4.0. "
    f"Nguồn: https://universe.roboflow.com/{ROBOFLOW_WORKSPACE}/{ROBOFLOW_PROJECT}",
    "- **Tomato Leaf Disease Detection (Zenodo)** (`leaf_zenodo`): CC BY 4.0 — "
    "CHƯA gộp vào dataset chính (chỉ audit), xem `zenodo_audit.md`. "
    "Nguồn: https://zenodo.org/records/20004230",
])
(MANIFESTS / "licenses.md").write_text(licenses_md, encoding="utf-8")
print(licenses_md)

report_lines = [
    "# tomato_leaf_disease_v1 — Dataset Report",
    f"Sinh tự động bởi `01_build_tomato_leaf_disease_v1.ipynb` — {pd.Timestamp.now().date()}",
    "",
    "## Convert leaf_roboflow",
    str(stats),
    "",
    "## Phân bố class + negative theo split",
    class_dist_df.to_string(index=False),
    "",
    "## Trùng lặp (trong leaf_roboflow)",
    f"- Tổng ảnh (positive + negative): {n}",
    f"- Ảnh nằm trong nhóm trùng/near-duplicate: {len(dup_report)}",
    f"- Số nhóm trùng: {dup_report['duplicate_group_id'].nunique() if len(dup_report) else 0}",
    "",
    "## Audit leaf_zenodo (chưa gộp)",
    f"- {len(zenodo_images)} ảnh, trùng SHA-256 với leaf_roboflow: {exact_overlap} "
    f"({100 * exact_overlap / max(len(zenodo_sha), 1):.1f}%). Chi tiết: `zenodo_audit.md`.",
    "",
    "## Cảnh báo cần xử lý thủ công",
    f"- `rejected_images.csv`: {len(rejected_rows)} ảnh bị loại hoàn toàn vì chỉ có box bệnh ngoài "
    "phạm vi MVP (không phải Healthy, không có box mục tiêu) — không dùng làm negative để tránh dạy sai model.",
    "- Ảnh Healthy trở thành negative sample (label rỗng) theo đúng thiết kế detection của dự án — "
    "không tạo class `healthy` riêng.",
    "- `leaf_zenodo` KHÔNG có trong `data.yaml`/`PROCESSED` — chỉ audit, chờ quyết định gộp thủ công.",
]
report = "\n".join(report_lines)
(MANIFESTS / "dataset_report.md").write_text(report, encoding="utf-8")
print("\n" + report)

### Bước 9 — Xác nhận trực quan (bắt buộc trước khi tin tưởng dataset)
Vẽ bounding box lên ảnh mẫu dương, và hiển thị riêng vài ảnh **negative** (phải là lá khỏe thật, không có box) để xác nhận bước lọc `Healthy` ở Bước 4 đúng.

In [ ]:
def show_samples(split, only="any", n=6, title=""):
    """only: 'any' | 'positive' (có ít nhất 1 box) | 'negative' (label rỗng)."""
    images_dir = PROCESSED / split / "images"
    labels_dir = PROCESSED / split / "labels"
    all_imgs = sorted(images_dir.glob("*"))

    def label_lines(img_path):
        lp = labels_dir / (img_path.stem + ".txt")
        if not lp.exists():
            return []
        return [l for l in lp.read_text().splitlines() if l.strip()]

    if only == "positive":
        all_imgs = [p for p in all_imgs if label_lines(p)]
    elif only == "negative":
        all_imgs = [p for p in all_imgs if not label_lines(p)]

    if not all_imgs:
        print(f"[{title}] Không có ảnh để hiển thị (only={only}).")
        return
    sample = random.Random(42).sample(all_imgs, min(n, len(all_imgs)))

    fig, axes = plt.subplots(1, len(sample), figsize=(3.2 * len(sample), 3.2))
    axes = [axes] if len(sample) == 1 else axes
    colors = ["lime", "orange", "red", "magenta"]
    for ax, img_path in zip(axes, sample):
        img = Image.open(img_path)
        w, h = img.size
        ax.imshow(img)
        for line in label_lines(img_path):
            cid, xc, yc, bw, bh = line.split()
            cid = int(cid)
            xc, yc, bw, bh = map(float, (xc, yc, bw, bh))
            xmin, ymin = (xc - bw / 2) * w, (yc - bh / 2) * h
            rect = patches.Rectangle((xmin, ymin), bw * w, bh * h, linewidth=1.5,
                                      edgecolor=colors[cid], facecolor="none")
            ax.add_patch(rect)
            ax.text(xmin, max(ymin - 4, 0), TARGET_CLASSES[cid], color=colors[cid], fontsize=7, weight="bold")
        ax.set_title(img_path.name, fontsize=6)
        ax.axis("off")
    fig.suptitle(title, fontsize=10)
    plt.tight_layout()
    save_path = MANIFESTS / f"audit_samples_{title.replace(' ', '_')}.png"
    plt.savefig(save_path, dpi=80, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print("Đã lưu:", save_path)


show_samples("train", only="positive", title="train_positive_boxes")
show_samples("train", only="negative", title="train_negative_healthy")
show_samples("val", only="positive", title="val_positive_boxes")
show_samples("test", only="positive", title="test_positive_boxes")

## Kết quả
- `tomato_leaf_disease_v1` hoàn chỉnh nằm ở `/kaggle/working/ai/datasets/processed/tomato_leaf_disease_v1/` với `train/`, `val/`, `test/`, và `data.yaml` (4 class, hỗ trợ ảnh negative label rỗng).
- Toàn bộ manifest (mapping, thống kê, license, báo cáo, audit Zenodo, ảnh audit) nằm ở `/kaggle/working/ai/datasets/manifests/`.

## Trước khi Save Version — checklist
- [ ] Bước 2: tên class gốc in ra khớp với 11 class công khai đã biết (không có class lạ bị âm thầm rơi vào OUT_OF_SCOPE ngoài ý muốn).
- [ ] Bước 9: ảnh `train_negative_healthy` thực sự là lá khỏe, không sót box nào.
- [ ] `dataset_report.md`: `images_kept_positive` + `images_kept_negative` hợp lý so với `images_total`; `images_rejected` không quá lớn.
- [ ] `rejected_images.csv`: xem qua để biết ảnh nào bị loại và vì sao.
- [ ] `zenodo_audit.md`: ghi nhận số liệu, chưa cần quyết định gộp ngay.

Sau khi hài lòng, bấm **Save Version → Save & Run All (Commit)** để `tomato_leaf_disease_v1` trở thành Kaggle Dataset độc lập.

## Bước tiếp theo
- (Khuyến nghị) `01b_dataset_report_leaf.ipynb`: đọc lại Kaggle Dataset đã Save Version, in thống kê + ảnh mẫu (giống `01b_dataset_report_ripeness.ipynb` của nhánh độ chín) để xác nhận nhanh mà không cần chạy lại toàn bộ pipeline.
- `03_train_leaf_baseline.ipynb` (chưa tạo): Add Input dataset vừa tạo, train YOLO nano (imgsz 640, epochs ~100, patience 20, seed 42) trên `train/val`, đánh giá trên `test`, theo đúng cấu hình baseline trong `TONG_HOP_YEU_CAU_VA_DE_XUAT_AI_SMART_GREENHOUSE.docx`.